# 大富豪 CPU 自己対局学習
ランタイムのタイプをGPUにしてから上から実行します。自己対局はTypeScriptの本番rules/authority、学習はPyTorch CUDAを使います。

In [ ]:
!git clone --depth 1 https://github.com/selmtoe/D.git /content/daifugo
%cd /content/daifugo
!corepack enable
!corepack prepare pnpm@11.19.0 --activate
!pnpm install --frozen-lockfile
import torch
assert torch.cuda.is_available(), 'GPUランタイムへ変更してください'
print(torch.__version__, torch.cuda.get_device_name(0))

In [ ]:
import os
os.environ['CPU_SELFPLAY_MATCHES'] = '120'
!pnpm ai:selfplay

In [ ]:
!python tools/ai/train.py --device cuda --epochs 60 --batch-size 256 --learning-rate 0.0002 --weight-decay 0.0001 --hidden-dim 128 --validation-fraction 0.2 --test-fraction 0.2 --run-name colab-selfplay-v3

In [ ]:
from pathlib import Path
import hashlib, subprocess, sys
checkpoint = Path('artifacts/ai/runs/colab-selfplay-v3/policy.pt')
digest = hashlib.sha256(checkpoint.read_bytes()).hexdigest()
output = Path('apps/web/src/ai/cpu-policy-colab.json')
subprocess.run([sys.executable, 'tools/ai/export_web.py', str(checkpoint), str(output), '--sha256', digest], check=True)
print('checkpoint SHA-256:', digest)

In [ ]:
from google.colab import files
files.download('apps/web/src/ai/cpu-policy-colab.json')
files.download('artifacts/ai/runs/colab-selfplay-v3/metrics.json')